Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import pickle
import xgboost as xgb
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_recall_curve,
    average_precision_score,
    roc_curve,
    roc_auc_score,
    classification_report,
    f1_score
)

In [ ]:
#train_df = pd.read_parquet("/content/drive/MyDrive/SB/SB_Project/classification_ring/data/processed/train.parquet")
train_df = pd.read_parquet('./classification_ring/data/processed/train.parquet')
#val_df = pd.read_parquet("/content/drive/MyDrive/SB/SB_Project/classification_ring/data/processed/val.parquet")
val_df = pd.read_parquet('./classification_ring/data/processed/val.parquet')

label_cols = ['HBOND', 'VDW', 'IONIC', 'PIPISTACK', 'PICATION', 'SSBOND', 'PIHBOND']
target_names = label_cols

num_features = [
    's_rsa', 's_phi', 's_psi', 's_a1', 's_a2', 's_a3', 's_a4', 's_a5',
    't_rsa', 't_phi', 't_psi', 't_a1', 't_a2', 't_a3', 't_a4', 't_a5'
]

cat_features = ['s_ss8', 's_3di_letter', 't_ss8', 't_3di_letter']
feature_cols = num_features + cat_features

X_train = train_df[feature_cols]
Y_train = train_df[label_cols]
X_val = val_df[feature_cols]
Y_val = val_df[label_cols]


In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])


In [ ]:
xgb_base = xgb.XGBClassifier(
    n_estimators=150,        # Number of boosting rounds
    max_depth=6,             # Maximum tree depth
    learning_rate=0.1,       # Step size shrinkage
    tree_method='hist',      # Optimized training method for large tabular datasets
    random_state=42,
    n_jobs=-1                # Use all available CPU cores
)

xgb_model = MultiOutputClassifier(xgb_base)

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
], verbose=True)

print("XGBoost Pipeline built successfully.")

XGBoost Pipeline built successfully.


Training

In [ ]:
train_model_bool = True  # Toggle to True to perform training
save_model_bool = True

if train_model_bool:
    print("Training XGBoost Pipeline...")
    xgb_pipeline.fit(X_train, Y_train)
    print("Training complete!")

    if save_model_bool:
        os.makedirs("classification_ring/models/xgb_classifier", exist_ok=True)
        model_path = "classification_ring/models/xgb_classifier/xgb_model.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(xgb_pipeline, f)
        print(f"Model saved to {model_path}")

Training XGBoost Pipeline...
[Pipeline] ...... (step 1 of 2) Processing preprocessor, total=   5.7s
[Pipeline] ........ (step 2 of 2) Processing classifier, total= 4.3min
Training complete!
Model saved to classification_ring/models/xgb_classifier/xgb_model.pkl


In [ ]:
def get_positive_class_scores(model, X):
    y_proba = model.predict_proba(X)
    if isinstance(y_proba, list):
        y_scores = np.column_stack([proba[:, 1] for proba in y_proba])
    else:
        y_scores = y_proba[:, 1]
    return y_scores

# Calculate predictions using validation thresholds
y_scores = get_positive_class_scores(xgb_pipeline, X_val)

In [ ]:
# Extract feature importance rankings across estimators
fitted_preprocessor = xgb_pipeline.named_steps["preprocessor"]
transformed_names = fitted_preprocessor.get_feature_names_out()

# Inspect feature importance for the first interaction type (e.g., HBOND)
first_class_tree = xgb_pipeline.named_steps["classifier"].estimators_[0]
importances = first_class_tree.feature_importances_

importance_df = pd.DataFrame({
    "feature": transformed_names,
    "importance": importances
}).sort_values("importance", ascending=False)
print(importance_df.head(10))

                feature  importance
6             num__s_a4    0.065852
74  cat__t_3di_letter_T    0.055070
14            num__t_a4    0.041832
70  cat__t_3di_letter_P    0.037971
34  cat__s_3di_letter_J    0.033311
12            num__t_a2    0.026267
40  cat__s_3di_letter_P    0.025723
43  cat__s_3di_letter_S    0.024038
62  cat__t_3di_letter_H    0.023951
44  cat__s_3di_letter_T    0.023772
